In [6]:
!unzip audiorecord.zip

Archive:  audiorecord.zip
   creating: audiorecord/
  inflating: audiorecord/DMF.m4a     
  inflating: audiorecord/EOT.m4a     
  inflating: audiorecord/kb1.m4a     
  inflating: audiorecord/LICS.m4a    
  inflating: audiorecord/nbdid.m4a   
  inflating: audiorecord/OOVB.m4a    
  inflating: audiorecord/PC.m4a      
  inflating: audiorecord/PE.m4a      


In [2]:
!pip install faster-whisper librosa pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 85.9 MB/s eta 0:00:00


Normal execution of the faster whisper

In [7]:
import os
import time
import pandas as pd
import librosa
from faster_whisper import WhisperModel

# 1. Configuration & Model Loading
AUDIO_DIR = "/content/audiorecord"
MODEL_SIZE = "large-v3"
DEVICE = "cuda"
COMPUTE_TYPE = "float16"

print(f"Loading Whisper model '{MODEL_SIZE}' on {DEVICE} ({COMPUTE_TYPE})...")
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

# 2. Supported Audio Extensions
VALID_EXTENSIONS = ('.mp3', '.wav', '.m4a', '.flac', '.ogg')

results = []

if not os.path.exists(AUDIO_DIR):
    print(f"Error: Directory '{AUDIO_DIR}' does not exist.")
else:
    audio_files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(VALID_EXTENSIONS)]
    print(f"Found {len(audio_files)} audio file(s) in {AUDIO_DIR}\n")

    for file_name in audio_files:
        audio_path = os.path.join(AUDIO_DIR, file_name)

        # Audio Duration Metric
        try:
            audio_duration = librosa.get_duration(path=audio_path)
        except Exception:
            audio_duration = 0.0

        # Timing Metric: Measure transcription wall-clock time
        start_time = time.time()

        segments, info = model.transcribe(audio_path, beam_size=5)
        hypothesis = " ".join([segment.text for segment in segments]).strip()

        elapsed_time = time.time() - start_time

        # Real-Time Factor (RTF)
        # RTF = Inference Time / Audio Duration (Values < 1.0 mean faster than real-time)
        rtf = elapsed_time / audio_duration if audio_duration > 0 else 0.0

        results.append({
            "File": file_name,
            "Detected Language": info.language,
            "Language Prob": round(info.language_probability, 2),
            "Duration (s)": round(audio_duration, 2),
            "Inference Time (s)": round(elapsed_time, 2),
            "RTF": round(rtf, 3),
            "Transcription": hypothesis
        })

# 3. Display Results Table in Colab
df = pd.DataFrame(results)

if not df.empty:
    print("\n=== Timing & Speed Metrics Summary ===")
    display(df[["File", "Duration (s)", "Inference Time (s)", "RTF"]])

    # Save detailed results to CSV
    output_csv = "/content/transcription_results.csv"
    df.to_csv(output_csv, index=False)
    print(f"\nSaved transcriptions and timing details to: {output_csv}")
else:
    print("\nNo audio files were processed, so no results to display or save.")

Loading Whisper model 'large-v3' on cuda (float16)...
Found 8 audio file(s) in /content/audiorecord



/tmp/ipykernel_1328/199088855.py:32: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  audio_duration = librosa.get_duration(path=audio_path)
/tmp/ipykernel_1328/199088855.py:32: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  audio_duration = librosa.get_duration(path=audio_path)
/tmp/ipykernel_1328/199088855.py:32: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  audio_duration = librosa.get_duration(path=audio_path)
/tmp/ipykernel_1328/199088855.py:32: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  audio_duration = librosa.get_duration(path=audio_path)
/tmp/ipykernel_1328/199088855.py


=== Timing & Speed Metrics Summary ===


,File,Duration (s),Inference Time (s),RTF
0,PC.m4a,6.9,2.71,0.392
1,kb1.m4a,8.7,1.89,0.217
2,LICS.m4a,9.3,1.60,0.172
3,PE.m4a,10.0,1.76,0.176
4,nbdid.m4a,8.8,1.00,0.114
5,EOT.m4a,10.2,1.18,0.116
6,DMF.m4a,9.6,1.52,0.159
7,OOVB.m4a,9.1,1.70,0.186



Saved transcriptions and timing details to: /content/transcription_results.csv


Faster whisper with prompt and hot words boosting

In [9]:
import os
import time
import pandas as pd
from faster_whisper import WhisperModel

# ------------------------------------------------------------------
# 1. Configuration & Models
# ------------------------------------------------------------------
AUDIO_DIR = "/content/audiorecord"
MODEL_SIZE = "large-v3"
DEVICE = "cuda"
COMPUTE_TYPE = "float16"

# Relevant Hinglish brand names, technical terms, and numeric phrases from Part 6 test scripts
BOOST_KEYWORDS = [
    "SraVaani Pro",
    "UPI",
    "SurgeX",
    "Khatabook",
    "bahi-khata",
    "25th September 2026",
    "3500 rupees",
    "cashback claim",
    "refund process",
    "manager",
    "492810",
    "984450"
]

# Convert keywords list to format strings
HOTWORDS_STR = " ".join(BOOST_KEYWORDS)                               # Space-separated string for faster-whisper hotwords
PROMPT_STR = f"The following Hinglish conversation mentions terms such as: {', '.join(BOOST_KEYWORDS)}."  # Context prompt string

print("Loading model...")
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

# ------------------------------------------------------------------
# 2. Evaluation Helper Functions
# ------------------------------------------------------------------
def analyze_keywords(text: str, target_keywords: list):
    """Counts how many target keywords are present in the predicted text."""
    text_lower = text.lower()
    found_keywords = [kw for kw in target_keywords if kw.lower() in text_lower]
    return found_keywords

# ------------------------------------------------------------------
# 3. Processing Audio Files Across 3 Configurations
# ------------------------------------------------------------------
VALID_EXT = ('.mp3', '.wav', '.m4a', '.flac')
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(VALID_EXT)] if os.path.exists(AUDIO_DIR) else []

results = []

for file_name in audio_files:
    audio_path = os.path.join(AUDIO_DIR, file_name)

    # Defined test configurations
    configs = {
        "1. Baseline (No Boost)": {"hotwords": None, "initial_prompt": None},
        "2. Hotwords Enabled":    {"hotwords": HOTWORDS_STR, "initial_prompt": None},
        "3. Initial Prompt":      {"hotwords": None, "initial_prompt": PROMPT_STR}
    }

    for cfg_name, kwargs in configs.items():
        start_time = time.time()

        # Transcribe with language set to 'hi' to ensure proper Hinglish code-switched decoding
        segments, _ = model.transcribe(
            audio_path,
            language="hi",
            beam_size=5,
            hotwords=kwargs["hotwords"],
            initial_prompt=kwargs["initial_prompt"]
        )

        hypothesis = " ".join([s.text for s in segments]).strip()
        elapsed = time.time() - start_time

        detected_kw = analyze_keywords(hypothesis, BOOST_KEYWORDS)

        results.append({
            "File": file_name,
            "Config": cfg_name,
            "Inference Time (s)": round(elapsed, 2),
            "Detected Keywords Count": len(detected_kw),
            "Detected Keywords": ", ".join(detected_kw),
            "Transcript": hypothesis
        })

# ------------------------------------------------------------------
# 4. Display & Export Results
# ------------------------------------------------------------------
df = pd.DataFrame(results)

if not df.empty:
    print("\n=== Keyword Boosting Experiment Results ===")
    display(df[["File", "Config", "Inference Time (s)", "Detected Keywords Count", "Detected Keywords"]])

    # Pivot view to easily compare configurations per file side-by-side
    pivot_df = df.pivot(index="File", columns="Config", values=["Detected Keywords Count", "Transcript"])
    display(pivot_df)

    df.to_csv("/content/keyword_boosting_results.csv", index=False)
else:
    print("\nNo audio files were processed, so no results to display or save.")

Loading model...

=== Keyword Boosting Experiment Results ===


,File,Config,Inference Time (s),Detected Keywords Count,Detected Keywords
0,PC.m4a,1. Baseline (No Boost),1.28,0,
1,PC.m4a,2. Hotwords Enabled,1.22,0,
2,PC.m4a,3. Initial Prompt,1.27,0,
3,kb1.m4a,1. Baseline (No Boost),1.71,0,
4,kb1.m4a,2. Hotwords Enabled,1.59,0,
5,kb1.m4a,3. Initial Prompt,1.76,0,
6,LICS.m4a,1. Baseline (No Boost),1.30,1,refund process
7,LICS.m4a,2. Hotwords Enabled,1.48,1,refund process
8,LICS.m4a,3. Initial Prompt,1.57,1,refund process
9,PE.m4a,1. Baseline (No Boost),1.47,0,


Detected Keywords Count                                        \
Config     1. Baseline (No Boost) 2. Hotwords Enabled 3. Initial Prompt   
File                                                                      
DMF.m4a                         0                   1                 0   
EOT.m4a                         1                   1                 1   
LICS.m4a                        1                   1                 1   
OOVB.m4a                        0                   1                 0   
PC.m4a                          0                   0                 0   
PE.m4a                          0                   0                 0   
kb1.m4a                         0                   0                 0   
nbdid.m4a                       1                   1                 1   

                                                  Transcript  \
Config                                1. Baseline (No Boost)   
File                                                           
DMF.m4a      मैने 25 सेटेंबर 2026 को 3500 रूपीज सेंट किया टी   
EOT.m4a                            मेरा एकॉंट नमबर है 492810   
LICS.m4a   मेरा refund process start हुआ है नहीं, status ...   
OOVB.m4a   मेरा सर्चेक्स कॉर्ड का कैसबेक क्लेव रिसेक्ट हो...   
PC.m4a            मेरा काटा बुक पर बाही कटा सिंक नहीं हो रहा   
PE.m4a     आहा आप लोग मेरा कॉल दो बार कट कर चुका हो मैनेज...   
kb1.m4a    मेरी स्रावनित्रों प्लान एको नहीं हुआ है। रुपिय...   
nbdid.m4a                            हाँ, order id है 984450   

                                                              \
Config                                   2. Hotwords Enabled   
File                                                           
DMF.m4a    मैंने 25th September 2026 को 3500 रूपीज सेंड क...   
EOT.m4a                            मेरा एकोंट नमबर है 492810   
LICS.m4a   मेरा refund process स्टार्ट हुआ है नहीं status...   
OOVB.m4a   मेरा सरचेक्स कॉर्ड का cashback claim रिसेक्ट ह...   
PC.m4a              मेरा काटबुक पर बाही कटा सिंक नहीं हो रहा   
PE.m4a     आहा आप लोग मेरा कॉल दो बार कट कर चुका हो मैनेज...   
kb1.m4a    मेरी स्रावनित प्लान एक्व नहीं हुआ है रुपिये से...   
nbdid.m4a                             हाँ order id है 984450   

                                                              
Config                                     3. Initial Prompt  
File                                                          
DMF.m4a    मैंने 25 सेटेंबर 2026 को 3500 रूपीज सेंड किया टी।  
EOT.m4a                          मेरा एक्कॉंट नंबर है 492810  
LICS.m4a   मेरा refund process स्टार्ट हुआ है नहीं। statu...  
OOVB.m4a   मेरा सर्चिक्स कॉर्ड का कैस बैक क्लेव रिसेक्ट ह...  
PC.m4a             मेरा काटबुक पर बाही कटा सिंक नहीं हो रहा।  
PE.m4a     आहा, आप लोग मेरा कॉल दो बार कट कर चुका हो, मैं...  
kb1.m4a    मेरी स्रावनित्व प्लान एक्व नहीं हुआ है। रुपिये...  
nbdid.m4a                            हाँ, order id है 984450

In [10]:
!pip install faster-whisper librosa pandas nemo_text_processing pynini -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 59.4 MB/s eta 0:00:00


integrating segment-level language identification (to route or filter wrong-language hallucinated segments) and NeMo Text Normalization (TN) / Inverse Text Normalization (ITN) alongside keyword boosting.

In [13]:
import os
import time
import pandas as pd
from faster_whisper import WhisperModel
from nemo_text_processing.text_normalization.normalize import Normalizer
from nemo_text_processing.inverse_text_normalization.inverse_normalize import InverseNormalizer

# ------------------------------------------------------------------
# 1. Initialize Whisper & NeMo TN/ITN Models
# ------------------------------------------------------------------
AUDIO_DIR = "/content/audiorecord"
MODEL_SIZE = "large-v3"
DEVICE = "cuda"
COMPUTE_TYPE = "float16"

# Supported target languages for your call center / domain routing
ALLOWED_LANGUAGES = ["hi", "en"]

# Keyword Boosting Configuration
BOOST_KEYWORDS = ["cashback claim", "refund process", "25th September 2026", "984450", "492810"]
HOTWORDS_STR = " ".join(BOOST_KEYWORDS)
PROMPT_STR = f"The following conversation mentions: {HOTWORDS_STR}."

print("Loading Whisper Model...")
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

print("Initializing NeMo TN and ITN Normalizers (Hindi & English)...")
# Inverse Text Normalization (Spoken words -> Digits, e.g., "पाँच हजार" -> "5000")
itn_hi = InverseNormalizer(lang="hi", input_case="lower_cased")
itn_en = InverseNormalizer(lang="en", input_case="lower_cased")

# Text Normalization (Digits -> Spoken words, e.g., "5000" -> "five thousand")
tn_hi = Normalizer(lang="hi", input_case="cased")
tn_en = Normalizer(lang="en", input_case="cased")

# ------------------------------------------------------------------
# 2. Segment-Level Processing & Normalization Logic
# ------------------------------------------------------------------
def process_audio_file(audio_path, use_boosting=True, apply_itn=True):
    start_time = time.time()

    # Transcribe audio with contextual boosting
    kwargs = {
        "beam_size": 5,
        "hotwords": HOTWORDS_STR if use_boosting else None,
        "initial_prompt": PROMPT_STR if use_boosting else None
    }

    segments, info = model.transcribe(audio_path, **kwargs)

    clean_segments = []
    filtered_segments = []

    for seg in segments:
        seg_lang = seg.language if hasattr(seg, 'language') and seg.language else info.language
        seg_text = seg.text.strip()

        # Segment-level Language ID Filtering: Filter out hallucinated non-target languages
        if seg_lang not in ALLOWED_LANGUAGES:
            filtered_segments.append({
                "start": round(seg.start, 2),
                "end": round(seg.end, 2),
                "detected_lang": seg_lang,
                "text": seg_text,
                "reason": "Language mismatch / potential hallucination"
            })
            continue

        # Apply NeMo Inverse Text Normalization (ITN) or Text Normalization (TN) based on detected language
        normalized_text = seg_text
        if apply_itn:
            try:
                if seg_lang == "hi":
                    normalized_text = itn_hi.inverse_normalize(seg_text, verbose=False)
                elif seg_lang == "en":
                    normalized_text = itn_en.inverse_normalize(seg_text, verbose=False)
            except Exception as e:
                # Fallback to raw segment text if normalizer encounters unsupported tokens
                normalized_text = seg_text

        clean_segments.append({
            "start": round(seg.start, 2),
            "end": round(seg.end, 2),
            "lang": seg_lang,
            "raw_text": seg_text,
            "normalized_text": normalized_text
        })

    elapsed = time.time() - start_time
    full_transcript = " ".join([s["normalized_text"] for s in clean_segments])

    return {
        "file": os.path.basename(audio_path),
        "primary_lang": info.language,
        "lang_prob": round(info.language_probability, 2),
        "inference_time": round(elapsed, 2),
        "full_transcript": full_transcript,
        "segments": clean_segments,
        "hallucinated_filtered": filtered_segments
    }

# ------------------------------------------------------------------
# 3. Execution & Evaluation
# ------------------------------------------------------------------
VALID_EXT = ('.mp3', '.wav', '.m4a', '.flac')
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(VALID_EXT)] if os.path.exists(AUDIO_DIR) else []

results = []
for file_name in audio_files:
    audio_path = os.path.join(AUDIO_DIR, file_name)
    res = process_audio_file(audio_path, use_boosting=True, apply_itn=True)
    results.append(res)

# ------------------------------------------------------------------
# 4. Display Results
# ------------------------------------------------------------------
summary_df = pd.DataFrame([{
    "File": r["file"],
    "Primary Lang": r["primary_lang"],
    "Lang Prob": r["lang_prob"],
    "Inference Time (s)": r["inference_time"],
    "Filtered Segments Count": len(r["hallucinated_filtered"]),
    "Final Transcript": r["full_transcript"]
} for r in results])

print("\n=== Pipeline Execution Summary ===")
display(summary_df)

Loading Whisper Model...
Initializing NeMo TN and ITN Normalizers (Hindi & English)...


 NeMo-text-processing :: INFO     :: Creating ClassifyFst grammars.
INFO:NeMo-text-processing:Creating ClassifyFst grammars.
 NeMo-text-processing :: INFO     :: Creating ClassifyFst grammars.
INFO:NeMo-text-processing:Creating ClassifyFst grammars.



=== Pipeline Execution Summary ===


,File,Primary Lang,Lang Prob,Inference Time (s),Filtered Segments Count,Final Transcript
0,PC.m4a,hi,0.86,2.23,0,मेरा काटा बुक पर बाही कटा सिंक नहीं हो रहा।
1,kb1.m4a,hi,0.36,2.83,0,मेरी स्रावनित्व प्लान एक्व नहीं हुआ है। रुपिये...
2,LICS.m4a,hi,0.95,2.63,0,मेरा refund process स्टार्ट हुआ है नहीं। statu...
3,PE.m4a,hi,0.93,2.65,0,आहा आप लोग मेरा कॉल २ बार कट कर चुका हो मैनेजर...
4,nbdid.m4a,hi,0.77,1.18,0,"हाँ, order id है 984450"
5,EOT.m4a,hi,0.56,1.45,0,मेरा एकाउंट नंबर है 492810
6,DMF.m4a,hi,0.54,1.95,0,मैंने 25 सेटेंबर 2026 को 3500 रूपीस सेंड किया टी
7,OOVB.m4a,hi,0.89,2.06,0,मेरा सर्चिक्स कॉर्ड का cashback claim रिसेक्ट ...


without language routing just applying inverse text normalization on all segments of transcriptions

In [17]:
import os
import time
import pandas as pd
from faster_whisper import WhisperModel
from nemo_text_processing.inverse_text_normalization.inverse_normalize import InverseNormalizer

# ------------------------------------------------------------------
# 1. Configuration & Model Loading
# ------------------------------------------------------------------
AUDIO_DIR = "/content/audiorecord"
MODEL_SIZE = "large-v3"
DEVICE = "cuda"
COMPUTE_TYPE = "float16"

# Keyword Boosting Setup
BOOST_KEYWORDS = ["cashback claim", "refund process", "25th September 2026", "984450", "492810"]
HOTWORDS_STR = " ".join(BOOST_KEYWORDS)
PROMPT_STR = f"The following conversation mentions: {HOTWORDS_STR}."

print("Loading Whisper Model...")
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

# ------------------------------------------------------------------
# 2. Initialize NeMo ITN Instances for Multiple Languages
# ------------------------------------------------------------------
print("Initializing NeMo Inverse Text Normalizers...")

# Dictionary of available normalizers per language code
itn_normalizers = {}

try:
    itn_normalizers["hi"] = InverseNormalizer(lang="hi", input_case="lower_cased")
except Exception as e:
    print(f"Warning: Failed to load Hindi ITN: {e}")

try:
    itn_normalizers["en"] = InverseNormalizer(lang="en", input_case="lower_cased")
except Exception as e:
    print(f"Warning: Failed to load English ITN: {e}")


def apply_nemo_itn(text: str, lang_code: str) -> str:
    """
    Applies NeMo ITN without language filtering constraints.
    Falls back gracefully to raw text if normalizer isn't available for the language or fails.
    """
    if not text.strip():
        return text

    # Select normalizer matching the language code, or default to English
    normalizer = itn_normalizers.get(lang_code, itn_normalizers.get("en"))

    if normalizer is None:
        return text

    try:
        return normalizer.inverse_normalize(text, verbose=False)
    except Exception:
        # Return original text if normalization fails or encounters unsupported tokens
        return text

# ------------------------------------------------------------------
# 3. Transcribe & Normalize All Segments (No Language Constraints)
# ------------------------------------------------------------------
def process_audio_unconstrained(audio_path):
    start_time = time.time()

    # Transcribe audio with keyword boosting enabled
    segments, info = model.transcribe(
        audio_path,
        beam_size=5,
        hotwords=HOTWORDS_STR,
        initial_prompt=PROMPT_STR
    )
    print(segments,info)

    processed_segments = []

    for seg in list(segments):
        # Retrieve language detected for the segment or default to audio language
        seg_lang = getattr(seg, 'language', None) or info.language
        raw_text = seg.text.strip()

        # Apply ITN directly without discarding any segment
        itn_text = apply_nemo_itn(raw_text, seg_lang)

        processed_segments.append({
            "start": round(seg.start, 2),
            "end": round(seg.end, 2),
            "detected_lang": seg_lang,
            "raw_text": raw_text,
            "itn_text": itn_text
        })

    elapsed = time.time() - start_time
    full_itn_transcript = " ".join([s["itn_text"] for s in processed_segments])

    return {
        "file": os.path.basename(audio_path),
        "primary_lang": info.language,
        "lang_prob": round(info.language_probability, 2),
        "inference_time": round(elapsed, 2),
        "full_transcript": full_itn_transcript,
        "segments": processed_segments
    }

# ------------------------------------------------------------------
# 4. Execution & Output
# ------------------------------------------------------------------
VALID_EXT = ('.mp3', '.wav', '.m4a', '.flac')
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(VALID_EXT)] if os.path.exists(AUDIO_DIR) else []

results = []
for file_name in audio_files:
    audio_path = os.path.join(AUDIO_DIR, file_name)
    res = process_audio_unconstrained(audio_path)
    results.append(res)

summary_df = pd.DataFrame([{
    "File": r["file"],
    "Primary Language": r["primary_lang"],
    "Language Prob": r["lang_prob"],
    "Inference Time (s)": r["inference_time"],
    "Final ITN Transcript": r["full_transcript"]
} for r in results])

print("\n=== Pipeline Execution Summary (No Language Constraint) ===")
display(summary_df)

Loading Whisper Model...
Initializing NeMo Inverse Text Normalizers...


 NeMo-text-processing :: INFO     :: Creating ClassifyFst grammars.
INFO:NeMo-text-processing:Creating ClassifyFst grammars.


<generator object WhisperModel.generate_segments at 0x495627e0> TranscriptionInfo(language='hi', language_probability=0.85888671875, duration=6.9333125, duration_after_vad=6.9333125, all_language_probs=[('hi', 0.85888671875), ('ur', 0.07989501953125), ('en', 0.0195770263671875), ('sa', 0.007785797119140625), ('la', 0.006305694580078125), ('pa', 0.005832672119140625), ('cy', 0.004833221435546875), ('nn', 0.0041656494140625), ('si', 0.0017919540405273438), ('km', 0.0015096664428710938), ('cs', 0.0013637542724609375), ('de', 0.0010623931884765625), ('fa', 0.0009083747863769531), ('bn', 0.0006341934204101562), ('ar', 0.0005297660827636719), ('te', 0.0004978179931640625), ('ta', 0.0004258155822753906), ('mr', 0.000415802001953125), ('gu', 0.00031876564025878906), ('jw', 0.00031638145446777344), ('sl', 0.00027489662170410156), ('ml', 0.00021827220916748047), ('sd', 0.00020194053649902344), ('ne', 0.00018894672393798828), ('es', 0.00017404556274414062), ('haw', 0.00015115737915039062), ('ps',

,File,Primary Language,Language Prob,Inference Time (s),Final ITN Transcript
0,PC.m4a,hi,0.86,2.40,मेरा काटा बुक पर बाही कटा सिंक नहीं हो रहा।
1,kb1.m4a,hi,0.36,2.72,मेरी स्रावनित्व प्लान एक्व नहीं हुआ है। रुपिये...
2,LICS.m4a,hi,0.95,2.51,मेरा refund process स्टार्ट हुआ है नहीं। statu...
3,PE.m4a,hi,0.93,2.62,आहा आप लोग मेरा कॉल २ बार कट कर चुका हो मैनेजर...
4,nbdid.m4a,hi,0.77,1.15,"हाँ, order id है 984450"
5,EOT.m4a,hi,0.56,1.43,मेरा एकाउंट नंबर है 492810
6,DMF.m4a,hi,0.54,2.06,मैंने 25 सेटेंबर 2026 को 3500 रूपीस सेंड किया टी
7,OOVB.m4a,hi,0.89,2.05,मेरा सर्चिक्स कॉर्ड का cashback claim रिसेक्ट ...


Above Inverse Normalizer didn't show any effect but here below cell we are using Normalizer which converting numbers to words

In [3]:
import os
import time
import pandas as pd
from faster_whisper import WhisperModel
from nemo_text_processing.text_normalization.normalize import Normalizer

# ------------------------------------------------------------------
# 1. Configuration & Model Loading
# ------------------------------------------------------------------
AUDIO_DIR = "/content/audiorecord"
MODEL_SIZE = "large-v3"
DEVICE = "cuda"
COMPUTE_TYPE = "float16"

# Keyword Boosting Setup
BOOST_KEYWORDS = ["cashback claim", "refund process", "25th September 2026", "984450", "492810"]
HOTWORDS_STR = " ".join(BOOST_KEYWORDS)
PROMPT_STR = f"The following conversation mentions: {HOTWORDS_STR}."

print("Loading Whisper Model...")
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

# ------------------------------------------------------------------
# 2. Initialize NeMo TN Instances for Multiple Languages
# ------------------------------------------------------------------
print("Initializing NeMo Text Normalizers...")

# Dictionary of available normalizers per language code
tn_normalizers = {}

try:
    tn_normalizers["hi"] = Normalizer(lang="hi", input_case="cased")
except Exception as e:
    print(f"Warning: Failed to load Hindi TN: {e}")

try:
    tn_normalizers["en"] = Normalizer(lang="en", input_case="cased")
except Exception as e:
    print(f"Warning: Failed to load English TN: {e}")


def apply_nemo_tn(text: str, lang_code: str) -> str:
    """
    Applies NeMo Text Normalization (TN) without language filtering constraints.
    Falls back gracefully to raw text if normalizer isn't available for the language or fails.
    """
    if not text.strip():
        return text

    # Select normalizer matching the language code, or default to English
    normalizer = tn_normalizers.get(lang_code, tn_normalizers.get("en"))

    if normalizer is None:
        return text

    try:
        return normalizer.normalize(text, verbose=False)
    except Exception:
        # Return original text if normalization fails or encounters unsupported tokens
        return text

# ------------------------------------------------------------------
# 3. Transcribe & Normalize All Segments (No Language Constraints)
# ------------------------------------------------------------------
def process_audio_unconstrained(audio_path):
    start_time = time.time()

    # Transcribe audio with keyword boosting enabled
    segments, info = model.transcribe(
        audio_path,
        beam_size=5,
        hotwords=HOTWORDS_STR,
        initial_prompt=PROMPT_STR
    )
    print(segments, info)

    processed_segments = []

    for seg in list(segments):
        # Retrieve language detected for the segment or default to audio language
        seg_lang = getattr(seg, 'language', None) or info.language
        raw_text = seg.text.strip()

        # Apply TN directly without discarding any segment
        tn_text = apply_nemo_tn(raw_text, seg_lang)

        processed_segments.append({
            "start": round(seg.start, 2),
            "end": round(seg.end, 2),
            "detected_lang": seg_lang,
            "raw_text": raw_text,
            "tn_text": tn_text
        })

    elapsed = time.time() - start_time
    full_tn_transcript = " ".join([s["tn_text"] for s in processed_segments])

    return {
        "file": os.path.basename(audio_path),
        "primary_lang": info.language,
        "lang_prob": round(info.language_probability, 2),
        "inference_time": round(elapsed, 2),
        "full_transcript": full_tn_transcript,
        "segments": processed_segments
    }

# ------------------------------------------------------------------
# 4. Execution & Output
# ------------------------------------------------------------------
VALID_EXT = ('.mp3', '.wav', '.m4a', '.flac')
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(VALID_EXT)] if os.path.exists(AUDIO_DIR) else []

results = []
for file_name in audio_files:
    audio_path = os.path.join(AUDIO_DIR, file_name)
    res = process_audio_unconstrained(audio_path)
    results.append(res)

summary_df = pd.DataFrame([{
    "File": r["file"],
    "Primary Language": r["primary_lang"],
    "Language Prob": r["lang_prob"],
    "Inference Time (s)": r["inference_time"],
    "Final TN Transcript": r["full_transcript"]
} for r in results])

print("\n=== Pipeline Execution Summary (No Language Constraint) ===")
display(summary_df)

Loading Whisper Model...
Initializing NeMo Text Normalizers...


 NeMo-text-processing :: INFO     :: Creating ClassifyFst grammars.
INFO:NeMo-text-processing:Creating ClassifyFst grammars.


<generator object WhisperModel.generate_segments at 0xe3c40270> TranscriptionInfo(language='hi', language_probability=0.85888671875, duration=6.9333125, duration_after_vad=6.9333125, all_language_probs=[('hi', 0.85888671875), ('ur', 0.07989501953125), ('en', 0.0195770263671875), ('sa', 0.007785797119140625), ('la', 0.006305694580078125), ('pa', 0.005832672119140625), ('cy', 0.004833221435546875), ('nn', 0.0041656494140625), ('si', 0.0017919540405273438), ('km', 0.0015096664428710938), ('cs', 0.0013637542724609375), ('de', 0.0010623931884765625), ('fa', 0.0009083747863769531), ('bn', 0.0006341934204101562), ('ar', 0.0005297660827636719), ('te', 0.0004978179931640625), ('ta', 0.0004258155822753906), ('mr', 0.000415802001953125), ('gu', 0.00031876564025878906), ('jw', 0.00031638145446777344), ('sl', 0.00027489662170410156), ('ml', 0.00021827220916748047), ('sd', 0.00020194053649902344), ('ne', 0.00018894672393798828), ('es', 0.00017404556274414062), ('haw', 0.00015115737915039062), ('ps',

,File,Primary Language,Language Prob,Inference Time (s),Final TN Transcript
0,PC.m4a,hi,0.86,2.04,मेरा काटा बुक पर बाही कटा सिंक नहीं हो रहा।
1,kb1.m4a,hi,0.36,2.35,मेरी स्रावनित्व प्लान एक्व नहीं हुआ है। रुपिये...
2,LICS.m4a,hi,0.95,2.26,मेरा आर ई एफ यू एन डी पी आर ओ सी ई एस एस स्टार...
3,PE.m4a,hi,0.93,2.23,आहा आप लोग मेरा कॉल दो बार कट कर चुका हो मैनेज...
4,nbdid.m4a,hi,0.77,1.55,"हाँ, ओ आर डी ई आर आई डी है नौ लाख चौरासी हज़ार..."
5,EOT.m4a,hi,0.56,1.59,मेरा एकाउंट नंबर है चार नौ दो आठ एक शून्य
6,DMF.m4a,hi,0.54,1.89,मैंने पच्चीस सेटेंबर दो हज़ार छब्बीस को तीन हज...
7,OOVB.m4a,hi,0.89,1.92,मेरा सर्चिक्स कॉर्ड का सी ए एस एच बी ए सी के स...


YAMNET model for event classifier and END of TURN or silence calculation using endpoint timestamp from whisper. and also showing whisper average Log Probability

In [18]:
!pip install faster-whisper librosa pandas tensorflow tensorflow-hub -q

In [1]:
import os
import time
import pandas as pd
import numpy as np
import librosa
import tensorflow as tf
import tensorflow_hub as hub
from faster_whisper import WhisperModel

# ------------------------------------------------------------------
# 1. Load Whisper Model & YAMNet (Audio Event Classifier)
# ------------------------------------------------------------------
AUDIO_DIR = "/content/audiorecord"
MODEL_SIZE = "large-v3"
DEVICE = "cuda"
COMPUTE_TYPE = "float16"

print("Loading Whisper model...")
whisper_model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

print("Loading YAMNet Audio Event Tagger...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

# Load YAMNet Class Map
class_map_path = yamnet_model.class_map_path().numpy().decode('utf-8')
class_names = [line.split(',')[2].strip('"') for line in open(class_map_path).readlines()[1:]]

# ------------------------------------------------------------------
# 2. Audio Processing Pipeline
# ------------------------------------------------------------------
def classify_audio_events(audio_path):
    """
    Classifies background audio events (Music, Noise, Laughter, Speech).
    YAMNet expects 16kHz mono audio.
    """
    try:
        # Resample 8kHz telephony audio to 16kHz for YAMNet
        wav_data, sr = librosa.load(audio_path, sr=16000, mono=True)
        waveform = tf.convert_to_tensor(wav_data, dtype=tf.float32)

        scores, embeddings, spectrogram = yamnet_model(waveform)
        mean_scores = np.mean(scores.numpy(), axis=0)

        # Get top 3 audio event tags
        top_indices = np.argsort(mean_scores)[::-1][:3]
        top_tags = [f"{class_names[i]} ({mean_scores[i]:.2f})" for i in top_indices]
        return ", ".join(top_tags)
    except Exception as e:
        return f"Event tagging failed: {str(e)}"


def process_audio_file(audio_path, hotwords=None, prompt=None):
    start_time = time.time()

    # Measure audio duration
    try:
        duration = librosa.get_duration(path=audio_path)
    except Exception:
        duration = 0.0

    # 1. Transcribe with Whisper
    segments, info = whisper_model.transcribe(
        audio_path,
        beam_size=5,
        hotwords=hotwords,
        initial_prompt=prompt,
        word_timestamps=True
    )

    segment_list = list(segments)
    elapsed = time.time() - start_time

    # 2. Extract Confidence Metrics
    if segment_list:
        avg_logprobs = [seg.avg_logprob for seg in segment_list]
        no_speech_probs = [seg.no_speech_prob for seg in segment_list]

        overall_confidence = round(float(np.mean(avg_logprobs)), 3)
        max_no_speech = round(float(np.max(no_speech_probs)), 3)
        last_speech_end = segment_list[-1].end
    else:
        overall_confidence = -99.0
        max_no_speech = 1.0
        last_speech_end = 0.0

    # Flag low confidence transcription (avg_logprob < -0.8 or high no_speech_prob)
    is_unsure = (overall_confidence < -0.8) or (max_no_speech > 0.6)

    # 3. Calculate End-of-Turn / Silence Tail
    trailing_silence = max(0.0, duration - last_speech_end)
    has_finished_speaking = trailing_silence >= 0.8  # Speaker paused > 800ms at turn end

    # 4. Extract Audio Events with YAMNet
    audio_events = classify_audio_events(audio_path)

    full_transcript = " ".join([seg.text.strip() for seg in segment_list])

    return {
        "File": os.path.basename(audio_path),
        "Duration (s)": round(duration, 2),
        "Inference Time (s)": round(elapsed, 2),
        "Avg LogProb (Confidence)": overall_confidence,
        "No Speech Prob": max_no_speech,
        "STT Unsure Flag": "YES" if is_unsure else "NO",
        "Trailing Silence (s)": round(trailing_silence, 2),
        "Turn Finished?": "YES" if has_finished_speaking else "NO",
        "Audio Events (YAMNet)": audio_events,
        "Transcript": full_transcript
    }

# ------------------------------------------------------------------
# 3. Execution
# ------------------------------------------------------------------
VALID_EXT = ('.mp3', '.wav', '.m4a', '.flac')
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(VALID_EXT)] if os.path.exists(AUDIO_DIR) else []

results = []
for file_name in audio_files:
    audio_path = os.path.join(AUDIO_DIR, file_name)
    res = process_audio_file(audio_path)
    results.append(res)

df = pd.DataFrame(results)

print("\n=== STT Confidence, End-Of-Turn & Audio Event Summary ===")
display(df[["File", "Avg LogProb (Confidence)", "STT Unsure Flag", "Trailing Silence (s)", "Turn Finished?", "Audio Events (YAMNet)"]])

df.to_csv("/content/stt_confidence_and_events.csv", index=False)

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Loading Whisper model...
Loading YAMNet Audio Event Tagger...


/tmp/ipykernel_25314/3243301153.py:57: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  duration = librosa.get_duration(path=audio_path)
/tmp/ipykernel_25314/3243301153.py:38: UserWarning: PySoundFile failed. Trying audioread instead.
  wav_data, sr = librosa.load(audio_path, sr=16000, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_25314/3243301153.py:57: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  duration = librosa.get_duration(path=audio_path)
/tmp/ipykernel_25314/3243301153.py:38: UserWarning: PySoundFile failed. Trying audioread inste


=== STT Confidence, End-Of-Turn & Audio Event Summary ===


/tmp/ipykernel_25314/3243301153.py:38: UserWarning: PySoundFile failed. Trying audioread instead.
  wav_data, sr = librosa.load(audio_path, sr=16000, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


,File,Avg LogProb (Confidence),STT Unsure Flag,Trailing Silence (s),Turn Finished?,Audio Events (YAMNet)
0,PC.m4a,-0.119,NO,1.30,YES,"Speech\n (0.77), Noise\n (0.04), White noise\n..."
1,kb1.m4a,-0.184,NO,1.80,YES,"Speech\n (0.60), Water\n (0.07), Vehicle\n (0.06)"
2,LICS.m4a,-0.178,NO,1.92,YES,"Speech\n (0.82), White noise\n (0.04), Inside ..."
3,PE.m4a,-0.108,NO,1.90,YES,"Speech\n (0.80), Pink noise\n (0.05), Noise\n ..."
4,nbdid.m4a,-0.302,NO,2.80,YES,"Speech\n (0.71), Rustle\n (0.04), Wind\n (0.03)"
5,EOT.m4a,-0.232,NO,2.20,YES,"Speech\n (0.57), Rustle\n (0.16), Pink noise\n..."
6,DMF.m4a,-0.128,NO,1.82,YES,"Speech\n (0.83), Pink noise\n (0.05), Rustle\n..."
7,OOVB.m4a,-0.143,NO,2.68,YES,"Speech\n (0.72), Rustle\n (0.10), Pink noise\n..."


Replaced END of turn feature using silero VAD

In [2]:
import os
import time
import pandas as pd
import numpy as np
import torch
import librosa
import tensorflow as tf
import tensorflow_hub as hub
from faster_whisper import WhisperModel

# ------------------------------------------------------------------
# 1. Load Silero VAD, Whisper & YAMNet Models
# ------------------------------------------------------------------
AUDIO_DIR = "/content/audiorecord"
MODEL_SIZE = "large-v3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

print("Loading Silero VAD Model...")
vad_model, vad_utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False,
    onnx=False
)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = vad_utils

print(f"Loading Whisper model '{MODEL_SIZE}' on {DEVICE}...")
whisper_model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

print("Loading YAMNet Audio Event Tagger...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')
class_map_path = yamnet_model.class_map_path().numpy().decode('utf-8')
class_names = [line.split(',')[2].strip('"') for line in open(class_map_path).readlines()[1:]]

# ------------------------------------------------------------------
# 2. Silero VAD Turn-Taking / Endpoint Detection Logic
# ------------------------------------------------------------------
def analyze_turn_taking_silero(audio_path, min_silence_duration_ms=800):
    """
    Uses Silero VAD to extract speech timestamps and determine if
    the customer finished speaking based on trailing silence.
    """
    # Silero VAD requires 16kHz sampling rate
    wav = read_audio(audio_path, sampling_rate=16000)
    audio_duration = len(wav) / 16000.0

    # Get speech timestamps using Silero VAD
    speech_timestamps = get_speech_timestamps(
        wav,
        vad_model,
        sampling_rate=16000,
        min_silence_duration_ms=min_silence_duration_ms,
        threshold=0.5
    )

    if speech_timestamps:
        # End time of the very last detected speech segment
        last_speech_end = speech_timestamps[-1]['end'] / 16000.0
        trailing_silence = max(0.0, audio_duration - last_speech_end)
        # End-of-turn condition: trailing silence exceeds min silence threshold
        finished_speaking = trailing_silence >= (min_silence_duration_ms / 1000.0)
        num_speech_chunks = len(speech_timestamps)
    else:
        last_speech_end = 0.0
        trailing_silence = audio_duration
        finished_speaking = True
        num_speech_chunks = 0

    return {
        "audio_duration": round(audio_duration, 2),
        "last_speech_end": round(last_speech_end, 2),
        "trailing_silence_s": round(trailing_silence, 2),
        "finished_speaking": finished_speaking,
        "num_speech_chunks": num_speech_chunks,
        "speech_timestamps": speech_timestamps
    }

# ------------------------------------------------------------------
# 3. YAMNet Audio Event Tagging
# ------------------------------------------------------------------
def classify_audio_events(audio_path):
    try:
        wav_data, sr = librosa.load(audio_path, sr=16000, mono=True)
        waveform = tf.convert_to_tensor(wav_data, dtype=tf.float32)

        scores, embeddings, spectrogram = yamnet_model(waveform)
        mean_scores = np.mean(scores.numpy(), axis=0)

        top_indices = np.argsort(mean_scores)[::-1][:3]
        top_tags = [f"{class_names[i]} ({mean_scores[i]:.2f})" for i in top_indices]
        return ", ".join(top_tags)
    except Exception as e:
        return f"Event tagging error: {str(e)}"

# ------------------------------------------------------------------
# 4. Integrated Processing Pipeline
# ------------------------------------------------------------------
def process_audio_file(audio_path, hotwords=None, prompt=None, min_silence_ms=800):
    start_time = time.time()

    # 1. Silero VAD End-of-Turn Analysis
    vad_res = analyze_turn_taking_silero(audio_path, min_silence_duration_ms=min_silence_ms)

    # 2. Transcribe with Whisper
    segments, info = whisper_model.transcribe(
        audio_path,
        beam_size=5,
        hotwords=hotwords,
        initial_prompt=prompt,
        word_timestamps=False
    )

    segment_list = list(segments)
    elapsed = time.time() - start_time

    # 3. Compute STT Confidence Metrics
    if segment_list:
        avg_logprobs = [seg.avg_logprob for seg in segment_list]
        no_speech_probs = [seg.no_speech_prob for seg in segment_list]

        overall_confidence = round(float(np.mean(avg_logprobs)), 3)
        max_no_speech = round(float(np.max(no_speech_probs)), 3)
    else:
        overall_confidence = -99.0
        max_no_speech = 1.0

    is_unsure = (overall_confidence < -0.8) or (max_no_speech > 0.6)

    # 4. Extract Paralinguistics with YAMNet
    audio_events = classify_audio_events(audio_path)

    full_transcript = " ".join([seg.text.strip() for seg in segment_list])

    return {
        "File": os.path.basename(audio_path),
        "Duration (s)": vad_res["audio_duration"],
        "Inference Time (s)": round(elapsed, 2),
        "VAD Speech Segments": vad_res["num_speech_chunks"],
        "Last Speech End (s)": vad_res["last_speech_end"],
        "Trailing Silence (s)": vad_res["trailing_silence_s"],
        "Turn Finished? (Silero)": "YES" if vad_res["finished_speaking"] else "NO",
        "STT Avg LogProb": overall_confidence,
        "STT Unsure Flag": "YES" if is_unsure else "NO",
        "Audio Events (YAMNet)": audio_events,
        "Transcript": full_transcript
    }

# ------------------------------------------------------------------
# 5. Execute Evaluation
# ------------------------------------------------------------------
VALID_EXT = ('.mp3', '.wav', '.m4a', '.flac')
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(VALID_EXT)] if os.path.exists(AUDIO_DIR) else []

results = []
for file_name in audio_files:
    audio_path = os.path.join(AUDIO_DIR, file_name)
    # Set min_silence_ms = 800ms for detecting speaker turn completion
    res = process_audio_file(audio_path, min_silence_ms=800)
    results.append(res)

df = pd.DataFrame(results)

print("\n=== Silero VAD, STT Confidence & Audio Events Summary ===")
display(df[[
    "File",
    "Duration (s)",
    "VAD Speech Segments",
    "Trailing Silence (s)",
    "Turn Finished? (Silero)",
    "STT Unsure Flag",
    "Audio Events (YAMNet)"
]])

# Save results
df.to_csv("/content/silero_vad_and_events_results.csv", index=False)

Loading Silero VAD Model...
The repository snakers4_silero-vad does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
Loading Whisper model 'large-v3' on cuda...
Loading YAMNet Audio Event Tagger...


/tmp/ipykernel_25314/3947410149.py:84: UserWarning: PySoundFile failed. Trying audioread instead.
  wav_data, sr = librosa.load(audio_path, sr=16000, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_25314/3947410149.py:84: UserWarning: PySoundFile failed. Trying audioread instead.
  wav_data, sr = librosa.load(audio_path, sr=16000, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_25314/3947410149.py:84: UserWarning: PySoundFile failed. Trying audioread instead.
  wav_data, sr = librosa.load(audio_path, 


=== Silero VAD, STT Confidence & Audio Events Summary ===


/tmp/ipykernel_25314/3947410149.py:84: UserWarning: PySoundFile failed. Trying audioread instead.
  wav_data, sr = librosa.load(audio_path, sr=16000, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


,File,Duration (s),VAD Speech Segments,Trailing Silence (s),Turn Finished? (Silero),STT Unsure Flag,Audio Events (YAMNet)
0,PC.m4a,6.93,1,1.11,YES,NO,"Speech\n (0.77), Noise\n (0.04), White noise\n..."
1,kb1.m4a,8.77,1,1.70,YES,NO,"Speech\n (0.60), Water\n (0.07), Vehicle\n (0.06)"
2,LICS.m4a,9.32,2,1.64,YES,NO,"Speech\n (0.82), White noise\n (0.04), Inside ..."
3,PE.m4a,10.07,1,1.69,YES,NO,"Speech\n (0.80), Pink noise\n (0.05), Noise\n ..."
4,nbdid.m4a,8.81,1,2.09,YES,NO,"Speech\n (0.71), Rustle\n (0.04), Wind\n (0.03)"
5,EOT.m4a,10.22,2,1.58,YES,NO,"Speech\n (0.57), Rustle\n (0.16), Pink noise\n..."
6,DMF.m4a,9.64,1,1.61,YES,NO,"Speech\n (0.83), Pink noise\n (0.05), Rustle\n..."
7,OOVB.m4a,9.17,1,2.68,YES,NO,"Speech\n (0.72), Rustle\n (0.10), Pink noise\n..."


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
